# NBFM Speaker Recognition — KV260 FPGA Deployment

**Inspired by hls4ml tutorial `part7b_deployment.ipynb`**  
Runs on the Kria KV260 with PYNQ installed.

### Workflow
```
Host machine                       KV260
──────────────────────────────     ──────────────────────────────────────────
MFCC_CNN_hls4ml_compact.ipynb  →   kria_deployment.ipynb  (this notebook)
  kria_package/                      kria_package/
    X_test.npy                         X_test.npy
    y_test.npy              scp        y_test.npy
    y_keras_*.npy          ────►       y_keras_*.npy
    classes.npy                        classes.npy
  drivers/                           drivers/
  bitfiles/                          bitfiles/
```

### Models available

| MODEL key | Architecture | Accuracy |
|-----------|-------------|----------|
| `float32`  | Float32 weights | 91.67 % |
| `qat16bit` | QAT 16-bit weights | 91.26 % |
| `qat8bit`  | QAT 8-bit weights | 80.89 % |
| `qat4bit`  | QAT 4-bit weights | 55.08 % |


## ① Model Selection

**Change `MODEL` here to switch between implementations.**  
All other cells adapt automatically.

In [ ]:
# ── MODEL SELECTION ────────────────────────────────────────────────────────
# Options: 'float32'  'qat16bit'  'qat8bit'  'qat4bit'
MODEL = 'qat16bit'

# ── PATHS (adjust if you placed files elsewhere) ────────────────────────────
PKG_DIR      = 'kria_package'      # directory with .npy files
DRIVERS_DIR  = 'drivers'           # directory with driver_*.py
BITFILES_DIR = 'bitfiles'          # directory with sub-folders per model

# ── MODEL REGISTRY ─────────────────────────────────────────────────────────
import os

MODEL_CONFIGS = {
    'float32': {
        'label':        'Float32',
        'driver_module':'driver_float32',
        'driver_class': 'SpeakerRecogFloat32Overlay',
        'bitfile':       os.path.join(BITFILES_DIR, 'float32',  'design_1_wrapper.bit'),
        'keras_npy':     os.path.join(PKG_DIR, 'y_keras_float32.npy'),
        'accuracy':      0.9167,
    },
    'qat16bit': {
        'label':        'QAT 16-bit',
        'driver_module':'driver_qat_16bit',
        'driver_class': 'SpeakerRecogQAT16Overlay',
        'bitfile':       os.path.join(BITFILES_DIR, 'qat_16bit', 'design_1_wrapper.bit'),
        'keras_npy':     os.path.join(PKG_DIR, 'y_keras_qat16bit.npy'),
        'accuracy':      0.9126,
    },
    'qat8bit': {
        'label':        'QAT 8-bit',
        'driver_module':'driver_qat_8bit',
        'driver_class': 'SpeakerRecogQAT8Overlay',
        'bitfile':       os.path.join(BITFILES_DIR, 'qat_8bit',  'design_1_wrapper.bit'),
        'keras_npy':     os.path.join(PKG_DIR, 'y_keras_qat8bit.npy'),
        'accuracy':      0.8089,
    },
    'qat4bit': {
        'label':        'QAT 4-bit',
        'driver_module':'driver_qat_4bit',
        'driver_class': 'SpeakerRecogQAT4Overlay',
        'bitfile':       os.path.join(BITFILES_DIR, 'qat_4bit',  'design_1_wrapper.bit'),
        'keras_npy':     os.path.join(PKG_DIR, 'y_keras_qat4bit.npy'),
        'accuracy':      0.5508,
    },
}

assert MODEL in MODEL_CONFIGS, f'Unknown MODEL "{MODEL}". Choose from: {list(MODEL_CONFIGS)}'
cfg = MODEL_CONFIGS[MODEL]
print(f'Selected model : {cfg["label"]}')
print(f'Bitfile        : {cfg["bitfile"]}')
print(f'Driver class   : {cfg["driver_class"]}')
print(f'Expected acc   : {cfg["accuracy"]:.2%}')


## ② Load Test Data

Files were generated by the `Export Test Dataset` cell in `MFCC_CNN_hls4ml_compact.ipynb`.

In [ ]:
import numpy as np

X_test  = np.load(os.path.join(PKG_DIR, 'X_test.npy'))           # (N, 20, 64, 2) float32
y_test  = np.load(os.path.join(PKG_DIR, 'y_test.npy'))           # (N,) int32
classes = np.load(os.path.join(PKG_DIR, 'classes.npy'), allow_pickle=True)  # (10,) str

# Keras reference predictions for the selected model
y_keras = np.load(cfg['keras_npy'])   # (N, 10) float32 softmax scores

N = len(X_test)
print(f'X_test  : {X_test.shape}  dtype={X_test.dtype}')
print(f'y_test  : {y_test.shape}  dtype={y_test.dtype}')
print(f'classes : {classes}')
print(f'y_keras : {y_keras.shape}  (reference predictions from Keras / host CPU)')
print(f'\nTest-set samples : {N}')
print(f'Class distribution:')
unique, counts = np.unique(y_test, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  [{c}] {classes[c]:8s}  {n:4d} samples')


## ③ Load FPGA Overlay

Downloads the bitstream to the KV260 PL fabric and sets up the AXI DMA channel.

In [ ]:
import sys, importlib

# Make sure drivers/ is on the module search path
if DRIVERS_DIR not in sys.path:
    sys.path.insert(0, DRIVERS_DIR)

# Dynamically import the selected driver
drv_mod   = importlib.import_module(cfg['driver_module'])
OverlayCls = getattr(drv_mod, cfg['driver_class'])

print(f'Loading overlay: {cfg["bitfile"]}')
overlay = OverlayCls(cfg['bitfile'])
print('Overlay loaded and PL programmed successfully.')

# Quick smoke test: single sample inference
x0     = X_test[0]
score0 = overlay.predict(x0)           # (10,) float32
pred0  = int(np.argmax(score0))
true0  = int(y_test[0])
print(f'\nSmoke test (sample 0):')
print(f'  Predicted : [{pred0}] {classes[pred0]}')
print(f'  True label: [{true0}] {classes[true0]}')
print(f'  Scores    : {np.round(score0, 4)}')


## ④ Full Test-Set Inference

Runs one DMA transfer per sample (the AXI Stream wrapper processes one MFCC frame at a time).  
Tracks per-sample latency and overall throughput.

Expected throughput on KV260 at 100 MHz:  
- Float32 / QAT 16-bit : ~21.9 µs latency → ~45 k inf/s (limited by DMA overhead)  
- QAT 4-bit            : ~14.6 µs latency → ~68 k inf/s

In [ ]:
from datetime import datetime

print(f'Running FPGA inference on {N} samples ({cfg["label"]})...')

y_hw       = np.zeros((N, len(classes)), dtype=np.float32)
latencies  = np.zeros(N, dtype=np.float64)

t_total_start = datetime.now()

for i, x in enumerate(X_test):
    t0          = datetime.now()
    y_hw[i]     = overlay.predict(x)
    latencies[i] = (datetime.now() - t0).total_seconds() * 1e6   # µs

    if (i + 1) % 100 == 0 or i == N - 1:
        elapsed = (datetime.now() - t_total_start).total_seconds()
        rate    = (i + 1) / elapsed
        print(f'  {i+1:5d}/{N}  elapsed={elapsed:.1f}s  rate={rate:.1f} inf/s', end='\r')

t_total = (datetime.now() - t_total_start).total_seconds()

print(f'\n\nDone.')
print(f'  Total time    : {t_total:.2f} s')
print(f'  Throughput    : {N / t_total:.1f} inf/s')
print(f'  Latency       : {latencies.mean():.1f} µs mean  '
      f'(min={latencies.min():.1f}  max={latencies.max():.1f}  p99={np.percentile(latencies,99):.1f})')


## ⑤ Results — Accuracy Comparison

Compares three inference paths (same approach as hls4ml tutorial part 7c):

| Path | Device |
|------|--------|
| Keras | Host CPU (float32 / QAT-aware) |
| FPGA  | KV260 PL, AXI Stream DMA, ap_fixed<16,6> I/O |

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_keras_cls = np.argmax(y_keras, axis=1)
y_hw_cls    = np.argmax(y_hw,    axis=1)

acc_keras = accuracy_score(y_test, y_keras_cls)
acc_hw    = accuracy_score(y_test, y_hw_cls)
acc_agree = accuracy_score(y_keras_cls, y_hw_cls)  # HW matches Keras

print('=' * 55)
print(f'  Model: {cfg["label"]}')
print('=' * 55)
print(f'  Keras CPU accuracy  : {acc_keras:.4f}  ({acc_keras:.2%})')
print(f'  FPGA KV260 accuracy : {acc_hw:.4f}  ({acc_hw:.2%})')
print(f'  HW/Keras agreement  : {acc_agree:.4f}  ({acc_agree:.2%})')
print(f'  Expected (training) : {cfg["accuracy"]:.2%}')
print('=' * 55)

print(f'\nPer-class report (FPGA):')
print(classification_report(y_test, y_hw_cls, target_names=classes))


In [ ]:
import matplotlib
matplotlib.use('Agg')    # no display needed on KV260; saves to file
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (y_pred_cls, title) in zip(axes, [
    (y_keras_cls, f'Keras CPU  ({acc_keras:.2%})'),
    (y_hw_cls,    f'FPGA KV260 ({acc_hw:.2%})'),
]):
    cm   = confusion_matrix(y_test, y_pred_cls)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'Confusion Matrix — {cfg["label"]}', fontsize=13)
plt.tight_layout()

plot_path = f'confusion_{MODEL}.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')


In [ ]:
# Per-class accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 4))

x = np.arange(len(classes))
w = 0.38

per_class_keras = np.array([
    np.mean(y_keras_cls[y_test == c] == c) for c in range(len(classes))
])
per_class_hw = np.array([
    np.mean(y_hw_cls[y_test == c] == c) for c in range(len(classes))
])

ax.bar(x - w/2, per_class_keras, w, label='Keras CPU', color='steelblue', alpha=0.8)
ax.bar(x + w/2, per_class_hw,    w, label='FPGA KV260', color='coral',    alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.axhline(acc_hw, color='grey', linestyle='--', linewidth=0.8, label=f'Overall HW acc ({acc_hw:.2%})')
ax.legend()
ax.set_title(f'Per-class Accuracy — {cfg["label"]}')
plt.tight_layout()

plot2_path = f'per_class_{MODEL}.png'
plt.savefig(plot2_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {plot2_path}')


In [ ]:
# Latency distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(latencies, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(latencies.mean(), color='red',    linestyle='--', label=f'mean {latencies.mean():.1f} µs')
ax.axvline(np.median(latencies), color='orange', linestyle='--', label=f'median {np.median(latencies):.1f} µs')
ax.set_xlabel('Latency per sample (µs)')
ax.set_ylabel('Count')
ax.set_title(f'Inference latency — {cfg["label"]} on KV260')
ax.legend()
plt.tight_layout()

plot3_path = f'latency_{MODEL}.png'
plt.savefig(plot3_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {plot3_path}')


## ⑥ Save FPGA Results

Save `y_hw_{MODEL}.npy` so results can be transferred back to the host for further analysis.

In [ ]:
hw_out = f'y_hw_{MODEL}.npy'
np.save(hw_out, y_hw)
print(f'Saved: {hw_out}  shape={y_hw.shape}')

# Print final summary
print()
print('=' * 55)
print(f'  SUMMARY — {cfg["label"]}')
print('=' * 55)
print(f'  Test samples     : {N}')
print(f'  Keras CPU acc    : {acc_keras:.4f}')
print(f'  FPGA KV260 acc   : {acc_hw:.4f}')
print(f'  HW/Keras agree   : {acc_agree:.4f}')
print(f'  Throughput       : {N / t_total:.1f} inf/s')
print(f'  Avg latency      : {latencies.mean():.1f} µs')
print('=' * 55)
print()
print(f'Copy results back to host:')
print(f'  scp kria@<kv260-ip>:~/speaker_recog/{hw_out} ./')


## ⑦ (Optional) Run All Models in Sequence

Iterates through all available bitfiles, collects accuracy and throughput, and prints a comparison table.  
Set `RUN_ALL = True` to execute.  Each model takes approximately as long as the single-model run above.

In [ ]:
RUN_ALL = False   # Set True to benchmark all available bitfiles

if RUN_ALL:
    import importlib
    all_results = []

    for mkey, mcfg in MODEL_CONFIGS.items():
        if not os.path.exists(mcfg['bitfile']):
            print(f'  {mcfg["label"]:12s}  SKIP (bitfile not found: {mcfg["bitfile"]})')
            continue
        if not os.path.exists(mcfg['keras_npy']):
            print(f'  {mcfg["label"]:12s}  SKIP (Keras ref not found: {mcfg["keras_npy"]})')
            continue

        print(f'\nLoading {mcfg["label"]}...')
        drv_mod_   = importlib.import_module(mcfg['driver_module'])
        OvlCls     = getattr(drv_mod_, mcfg['driver_class'])
        ovl        = OvlCls(mcfg['bitfile'])

        y_ref    = np.load(mcfg['keras_npy'])
        y_scores = np.zeros((N, len(classes)), dtype=np.float32)
        t0 = datetime.now()
        for i, x in enumerate(X_test):
            y_scores[i] = ovl.predict(x)
        dt = (datetime.now() - t0).total_seconds()

        acc   = accuracy_score(y_test, np.argmax(y_scores, axis=1))
        agree = accuracy_score(np.argmax(y_ref, axis=1), np.argmax(y_scores, axis=1))
        rate  = N / dt

        np.save(f'y_hw_{mkey}.npy', y_scores)
        all_results.append({
            'model':       mcfg['label'],
            'acc_keras':   accuracy_score(y_test, np.argmax(y_ref, axis=1)),
            'acc_fpga':    acc,
            'hw_agree':    agree,
            'throughput':  rate,
        })
        print(f'  acc={acc:.4f}  agree={agree:.4f}  throughput={rate:.1f} inf/s')
        del ovl  # free overlay before loading next

    print()
    print('=' * 78)
    print(f'  {"Model":<14}  {"Keras CPU":>10}  {"FPGA KV260":>10}  {"HW/Keras":>10}  {"Throughput":>14}')
    print('=' * 78)
    for r in all_results:
        print(f'  {r["model"]:<14}  {r["acc_keras"]:>10.4f}  {r["acc_fpga"]:>10.4f}'
              f'  {r["hw_agree"]:>10.4f}  {r["throughput"]:>12.1f} i/s')
    print('=' * 78)
else:
    print('RUN_ALL=False. Set to True to benchmark all bitfiles.')
